# CatBoost model

This notebook trains CatBoost, scores it with the same 5 folds as your other models, makes a submission and saves its predictions for blending in Stage 7.

**Why CatBoost:** like LightGBM, it builds boosted trees one after another, each correcting the errors of the ones before. It differs in two ways that can matter here:

- **It turns categories into numbers itself.** For each category, such as an occupation code, it uses the default rate of people with that code. To stop that leaking the answer, it works through the rows in a random order and only uses rows that came *before* the current one. This is called *ordered target statistics*.
- **Its trees are symmetric.** Every split at the same depth uses the same rule, so trees are simpler and less likely to overfit.

Run this notebook in your `dsde` environment, which already has CatBoost installed.

## 0. Reload shared code automatically

Your features live in `features.py`, which every model notebook shares. Python normally loads a file like that only once per kernel, so later edits would be ignored.

`autoreload` makes the notebook pick up your latest `features.py` each time a cell runs. Keep this as the first code cell.

In [18]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Project folders

This notebook lives in `notebooks/`, but the data, shared code and outputs are in other folders. This cell finds the project folder and imports their locations from `scripts/paths.py`:

| Name | Folder | Holds |
|---|---|---|
| `DATA` | `data/` | Competition files |
| `PREDS` | `preds_for_blend/` | Saved predictions for blending |
| `SUBMISSIONS` | `submissions/` | Files to upload |
| `LOGS` | `logs/` | CatBoost training logs |

It also adds `scripts/` to Python's search path, so `from features import ...` works.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "scripts"))

from paths import DATA, LOGS, PREDS, SUBMISSIONS

## 1. Imports

- `numpy` and `pandas` handle arrays and tables.
- `CatBoostClassifier` is the model.
- `StratifiedKFold` makes folds with the same default rate, and `roc_auc_score` scores them.
- `Path` lets us create a folder for saved predictions.

In [19]:
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from features import add_features

## 2. Settings

Keeping every setting in one cell makes experiments easy to change and repeat.

- `SEED = 42` matches your other notebooks, so CatBoost is tested on **exactly the same folds** as LightGBM. That makes a fold-by-fold comparison fair.
- `PRED_DIR` is where predictions are saved for blending later.

In [20]:
TARGET = "12-Month Default Flag"
SEED = 42
N_SPLITS = 5
PRED_DIR = PREDS
PRED_DIR.mkdir(exist_ok=True)

## 3. Load the data

- `X` holds the features. We drop `ID` because it isn't an applicant feature, and the target because that's the answer.
- `y` holds the 0/1 default flag.
- `X_test` gets the same columns as `X`, in the same order.

**Features:** `add_features()` from `features.py` adds your engineered columns. It runs on train and test in exactly the same way, so they always have the same columns. Here, `X_test` takes `X`'s column list, which keeps the order identical.


In [21]:
train = pd.read_csv(DATA / "train_dataset.csv")
test = pd.read_csv(DATA / "test_without_gt.csv")

X = add_features(train.drop(columns=["ID", TARGET]))
y = train[TARGET]
X_test = add_features(test)[X.columns]

print(X.shape, X_test.shape, f"default rate {y.mean():.4f}")

(68546, 26) (137091, 26) default rate 0.0968


## 4. Mark the category columns

These columns hold **codes, not quantities**. Occupation 1540 isn't "more" than occupation 140, so CatBoost needs to know to treat them as labels.

We convert them to text (`str`) because CatBoost requires category columns to be text or whole numbers, and text makes our intent clear. All other columns, such as age and income, stay numeric.

Trees only care about the **order** of numeric values, so there's no need to scale them or log-transform them like you would for logistic regression. That also means extreme incomes do far less damage here.

In [22]:
cats = [
    "Application Channel", "Occupation Category", "Industry Category",
    "Employment Type Category", "Insurance Type", "Company Size Category",
    "Business Entity Type", "Residence Type", "Residence Ownership Status",
]

X_cb = X.copy()
X_test_cb = X_test.copy()
for c in cats:
    X_cb[c] = X_cb[c].astype(str)
    X_test_cb[c] = X_test_cb[c].astype(str)

## 5. Define the model

These settings come from trial 19 of `tuning_loop.py`, which scored 0.6520 mean CV AUC against 0.6486 for the previous settings.

| Setting | What it does |
|---|---|
| `iterations=3000` | The **maximum** number of trees. Early stopping usually stops far sooner, at about 200 trees here. |
| `learning_rate=0.06` | How much each tree contributes. |
| `depth=7` | Every tree has 7 levels of splits. |
| `l2_leaf_reg=31.09` | A strong penalty that shrinks leaf values, so no single tree gets too confident. It balances the deeper trees. |
| `random_strength=0.2048` | How much random noise is added when scoring splits. This is low, so splits follow the data closely. |
| `subsample=0.7916` | Each tree trains on a random 79% of the rows. |
| `rsm=0.9794` | Each split considers a random 98% of the features. |
| `one_hot_max_size=20` | Categories with up to 20 values are one-hot encoded instead of target-encoded. All 9 of our category columns have 18 values or fewer, so they're all one-hot here. |
| `eval_metric="AUC"` | Tracks AUC on the validation fold, which early stopping uses. |
| `cat_features=cats` | Tells CatBoost which columns are categories. |
| `thread_count=-1` | Uses all your CPU cores. |
| `verbose=0` | Hides the per-tree training log. |

`make_catboost()` returns a **fresh, untrained** model each time, so nothing carries over between folds.

In [23]:
def make_catboost(iterations=3000):
    # Tuned settings: trial 19 from tuning_loop.py (CV AUC 0.6520).
    # Previous: learning_rate=0.03, depth=6, l2_leaf_reg=5 (CV AUC 0.6486).
    return CatBoostClassifier(
        iterations=iterations,
        learning_rate=0.06,
        depth=7,
        l2_leaf_reg=31.09,
        random_strength=0.2048,
        subsample=0.7916,
        rsm=0.9794,
        one_hot_max_size=20,
        eval_metric="AUC",
        cat_features=cats,
        random_seed=SEED,
        thread_count=-1,
        verbose=0,
        train_dir=str(LOGS / "catboost_info"),
    )

## 6. Cross-validate with early stopping

This is your `evaluate()` loop with one addition: **early stopping**.

- `eval_set=(X_val, y_val)` lets CatBoost check the validation AUC after every tree.
- `early_stopping_rounds=200` stops training if AUC hasn't improved for 200 trees, then keeps the best number of trees.
- `model.get_best_iteration()` tells us how many trees that was. We'll use it to size the final model.

**One caution:** the validation fold now helps pick the number of trees, so these scores are slightly optimistic. The effect is usually small, but keep it in mind when you compare against LightGBM run without early stopping.

This cell takes a few minutes, because it trains 5 models.

In [ ]:
cv = StratifiedKFold(N_SPLITS, shuffle=True, random_state=SEED)
oof = np.zeros(len(y))
scores, best_iters = [], []

for fold, (fit_idx, val_idx) in enumerate(cv.split(X_cb, y)):
    X_fit, y_fit = X_cb.iloc[fit_idx], y.iloc[fit_idx]
    X_val, y_val = X_cb.iloc[val_idx], y.iloc[val_idx]

    model = make_catboost()
    model.fit(X_fit, y_fit, eval_set=(X_val, y_val), early_stopping_rounds=200)

    oof[val_idx] = model.predict_proba(X_val)[:, 1]
    scores.append(roc_auc_score(y_val, oof[val_idx]))
    best_iters.append(model.get_best_iteration())
    print(f"Fold {fold}: AUC {scores[-1]:.4f}, best trees {best_iters[-1]}")

print(f"\nCatBoost CV AUC: {np.mean(scores):.4f} +/- {np.std(scores):.4f}")

## 7. Which features matter

`get_feature_importance()` shows how much each feature changed the predictions, on average. The values add up to 100.

Compare this with your EDA. Age, residence period, home phone and number of borrowings looked strong there. If something you didn't expect is near the top, that's worth investigating.

Importance shows what the model **used**, not what **causes** default. Related features, such as age and residence period, can also share credit.

In [ ]:
importance = pd.Series(
    model.get_feature_importance(), index=X_cb.columns
).sort_values(ascending=False)
importance.round(2).head(12)

## 8. Train the final model on all the data

The CV models each saw only 80% of the rows. For the submission, we train **one** model on all 100%, which usually predicts slightly better.

With no validation fold left, we can't use early stopping, so we set the tree count from CV:
- Take the average best iteration across the folds.
- Multiply by 1.1, because 25% more data usually supports a few more trees.

This is a common rule of thumb rather than an exact science.

In [ ]:
final_iters = int(np.mean(best_iters) * 1.1)
print("Trees for the final model:", final_iters)

final_model = make_catboost(iterations=final_iters)
final_model.fit(X_cb, y)
test_pred = final_model.predict_proba(X_test_cb)[:, 1]

Trees for the final model: 224


## 9. Write the submission

The competition's rules are strict: the same IDs, in the same order, with two columns and probabilities from 0 to 1.

The `assert` lines stop the notebook with an error if anything is wrong, so you don't upload a broken file and waste one of your 10 daily submissions.

In [27]:
sub = pd.read_csv(DATA / "submission_template_for_test.csv")
assert (sub["ID"].values == test["ID"].values).all(), "Row order doesn't match the template"

sub[TARGET] = test_pred

assert list(sub.columns) == ["ID", TARGET]
assert len(sub) == 137_091
assert sub[TARGET].between(0, 1).all() and sub[TARGET].notna().all()

sub.to_csv(SUBMISSIONS / "submission_catboost.csv", index=False)
print("Saved submissions/submission_catboost.csv")
sub[TARGET].describe()

Saved submission_catboost.csv


count    137091.000000
mean          0.096243
std           0.047654
min           0.010911
25%           0.062224
50%           0.086809
75%           0.119558
max           0.607959
Name: 12-Month Default Flag, dtype: float64

## 10. Save predictions for blending

Stage 7 blends models by combining their predictions. Saving them now means you won't need to retrain CatBoost later.

- `oof` has one honest prediction per **training** row, used to score a blend.
- `test_pred` has one prediction per **test** row, used to make the blended submission.

These files are derived from competition data, so delete the `preds_for_blend` folder with everything else after the competition.

In [28]:
np.save(PRED_DIR / "catboost_oof.npy", oof)
np.save(PRED_DIR / "catboost_test.npy", test_pred)

print(f"Overall OOF AUC: {roc_auc_score(y, oof):.4f}")

NameError: name 'oof' is not defined

## Your turn

1. What CV AUC did you get, and how does it compare with LightGBM on the same folds? Which model wins in more folds?
2. Change `depth` to 4 and then 8. Which scores best, and how does the best tree count change?
3. Upload `submission_catboost.csv`. How close is the public score to your CV score?